# Concrete Crack Detection - Explore & Train

End-to-end walkthrough of the project:

1. Load the Mendeley concrete crack dataset
2. Explore class balance and sample images
3. Fine-tune a ResNet18 backbone
4. Evaluate on a held-out validation split
5. Visualise a Grad-CAM heatmap on a few examples

Run this notebook from the project root so that the `src/` package imports work.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from src.dataset import CrackDataset, get_eval_transform, get_train_transform
from src.model import CLASS_NAMES, build_model

DATA_DIR = PROJECT_ROOT / 'data'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Dataset path exists:', DATA_DIR.exists())

## 1. Explore the dataset

The Mendeley set has 40,000 RGB images (20k positive, 20k negative) at 227x227. This is a very clean, balanced dataset, so we expect high accuracy out of the box and should be honest about that in the README.

In [ ]:
dataset = CrackDataset(DATA_DIR, transform=get_eval_transform())
labels = np.array([label for _, label in dataset.samples])
counts = np.bincount(labels, minlength=len(CLASS_NAMES))
for name, count in zip(CLASS_NAMES, counts):
    print(f'{name:<10s}: {count}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, target in enumerate([0, 1]):
    idxs = np.where(labels == target)[0][:4]
    for col, idx in enumerate(idxs):
        path, _ = dataset.samples[idx]
        axes[row, col].imshow(Image.open(path))
        axes[row, col].set_title(CLASS_NAMES[target])
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 2. Train

For a reproducible run, prefer the CLI: `python -m src.train --data-dir data/ --epochs 5`.

This cell lets you kick off a short training run inline (useful on Colab).

In [ ]:
# Uncomment to run training from the notebook.
# !python -m src.train --data-dir {DATA_DIR} --epochs 5 --batch-size 64 --output {PROJECT_ROOT / 'models' / 'crack_classifier.pt'}

## 3. Evaluate a trained model

After training, load the saved weights and compute a confusion matrix and classification report on the validation split.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset, random_split

from src.model import load_model

CHECKPOINT = PROJECT_ROOT / 'models' / 'crack_classifier.pt'
if not CHECKPOINT.exists():
    print('No checkpoint yet. Train first.')
else:
    model = load_model(CHECKPOINT, device=DEVICE)

    eval_ds = CrackDataset(DATA_DIR, transform=get_eval_transform())
    generator = torch.Generator().manual_seed(42)
    n_total = len(eval_ds)
    n_val = int(0.2 * n_total)
    _, val_idx = random_split(range(n_total), [n_total - n_val, n_val], generator=generator)
    val_loader = DataLoader(Subset(eval_ds, list(val_idx)), batch_size=64, shuffle=False)

    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            preds.extend(model(x).argmax(dim=1).cpu().tolist())
            targets.extend(y.tolist())

    print(classification_report(targets, preds, target_names=CLASS_NAMES))
    cm = confusion_matrix(targets, preds)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot()
    plt.show()

## 4. Grad-CAM visual explanation

Pick a few sample images and visualise where the network looks when predicting *Crack*.

In [ ]:
from src.gradcam import compute_gradcam_overlay

if not CHECKPOINT.exists():
    print('Train the model first to see meaningful Grad-CAM.')
else:
    sample_paths = [p for _, _ in zip(range(4), dataset.samples)]
    sample_paths = [dataset.samples[i][0] for i in np.random.RandomState(0).choice(len(dataset), 4, replace=False)]

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for col, path in enumerate(sample_paths):
        img = Image.open(path).convert('RGB')
        overlay = compute_gradcam_overlay(img, model=model, class_index=1, device=DEVICE)
        axes[0, col].imshow(img)
        axes[0, col].set_title(path.parent.name)
        axes[0, col].axis('off')
        axes[1, col].imshow(overlay)
        axes[1, col].set_title('Grad-CAM')
        axes[1, col].axis('off')
    plt.tight_layout()
    plt.show()